In [1]:
import numpy as np
import torch
import torch.optim as optim
from tasks import CartPole
from state_pred_models import NextStateQuantileNetwork, quantile_loss, NextStateSinglePredNetwork, quantile_loss_median, mse_loss
from main_funcs import main_QRNN_MPC, main_50NN_MSENN_MPC
from ASNN import ReplayBuffer_ASNN, ActionSequenceNN, gaussian_nll_loss, categorical_cross_entropy_loss, train_ActionSequenceNN
from choose_action_QRNN import choose_action_func_QRNN
from choose_action_50NN_MSENN import choose_action_func_50NN_MSENN
from RndUniformGeneratedActionSequences import generate_random_action_sequences
from ShiftParticlesReplaceWithRandom import ShiftParticlesReplaceWithRandom_func
from GenerateParticlesUsingASNN import GenerateParticlesUsingASNN_func_50NN_MSENN
from setup import setup_class
from state_pred_models import train_QRNN, train_50NN_MSENN
from matplotlib.animation import FFMpegWriter
from sklearn.metrics import auc


ImportError: cannot import name 'CartPole' from 'tasks' (unknown location)

Test pendulum simulation

In [ ]:
## [test simulation] ##
sim_step = 100
delta_t = 0.05
pendulum = Pendulum()
for i in range(sim_step):
    pendulum.update(u=[2.0 * np.sin(i/5.0)], delta_t=delta_t) # u is the control input to the pendulum, [ torque[Nm] ]
pendulum.show_animation(interval_ms=delta_t*1000) # show animation

Experiment boolean variables


In [ ]:
EXPERIMENTS = {
    "ASNN_mid_QRNN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "ASNN_mid_50NN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "ASNN_mid_MSENN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),

    "ASNN_mid_QRNN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "ASNN_mid_50NN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "ASNN_mid_MSENN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),

    "basic_mid_QRNN_PF": dict(use_ASNN=False, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "basic_mid_50NN_PF": dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "basic_mid_MSENN_PF": dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),

    "basic_mid_QRNN_CEM": dict(use_ASNN=False, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "basic_mid_50NN_CEM": dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),
    "basic_mid_MSENN_CEM": dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=False, do_RS=False),

    "rnd_mid_QRNN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),
    "rnd_mid_50NN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),
    "rnd_mid_MSENN_PF": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),

    "rnd_mid_QRNN_CEM": dict(use_ASNN=True, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),
    "rnd_mid_50NN_CEM": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),
    "rnd_mid_MSENN_CEM": dict(use_ASNN=True, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=True, use_sampling=False, do_QRNN_step_rnd=True, do_RS=False),
    
    "RS_mid_QRNN":       dict(use_ASNN=False, use_mid=True, use_QRNN=True, use_50NN=False, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=True),
    "RS_mid_50NN":       dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=True, use_MSENN=False, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=True),
    "RS_mid_MSENN":      dict(use_ASNN=False, use_mid=True, use_QRNN=False, use_50NN=False, use_MSENN=True, use_CEM=False, use_sampling=False, do_QRNN_step_rnd=False, do_RS=True),
}


In [ ]:
method_name = "ASNN_mid_QRNN_PF"
# method_name = "ASNN_mid_50NN_PF"
# method_name = "ASNN_mid_MSENN_PF"
# method_name = "ASNN_mid_QRNN_CEM"
# method_name = "ASNN_mid_50NN_CEM"
# method_name = "ASNN_mid_MSENN_CEM"
# method_name = "basic_mid_QRNN_PF"
# method_name = "basic_mid_50NN_PF"
# method_name = "basic_mid_MSENN_PF"
# method_name = "basic_mid_QRNN_CEM"
# method_name = "basic_mid_50NN_CEM"
# method_name = "basic_mid_MSENN_CEM"
# method_name = "rnd_mid_QRNN_PF"
# method_name = "rnd_mid_50NN_PF"
# method_name = "rnd_mid_MSENN_PF"
# method_name = "rnd_mid_QRNN_CEM"
# method_name = "rnd_mid_50NN_CEM"
# method_name = "rnd_mid_MSENN_CEM"
# method_name = "RS_mid_QRNN"
# method_name = "RS_mid_50NN"
# method_name = "RS_mid_MSENN"

config = EXPERIMENTS[method_name]

# unpack flags
use_ASNN   = config["use_ASNN"]
use_mid    = config["use_mid"]
use_QRNN   = config["use_QRNN"]
use_50NN   = config["use_50NN"]
use_MSENN  = config["use_MSENN"]
use_CEM    = config["use_CEM"]
use_sampling = config["use_sampling"]
do_QRNN_step_rnd = config["do_QRNN_step_rnd"]
do_RS      = config["do_RS"]


In [ ]:
def compute_auc_and_std(data, nb_episodes):
    # data = np.load(filepath, allow_pickle=True)
    # all_returns = data['episode_rewards']  # shape: (n_seeds, n_episodes)
    # print("all_returns ", all_returns, "\n")
    
    # print("all_returns shape:", all_returns.shape, "\n")
    # aucs = []
    # for rewards in all_returns:
    #     returns = rewards[:nb_episodes]
    #     x = np.arange(len(returns))
    #     aucs.append(auc(x, returns))
    
    x = np.arange(len(data))
    aucs = auc(x,data)

    auc_mean = np.mean(aucs)
    auc_std = np.std(aucs)
    return auc_mean, auc_std

In [ ]:
pendulum_theta_dict = {}
pendulum_theta_dot_dict = {}

pendulum_theta_dict_auc_mean = {}
pendulum_theta_dict_auc_std = {}
pendulum_theta_dot_dict_auc_mean = {}
pendulum_theta_dot_dict_auc_std = {}

In [ ]:
for key, value in EXPERIMENTS.items():
    print(f"{key} : {value}")
    method_name = key
    config = value
    # unpack flags
    use_ASNN   = config["use_ASNN"]
    use_mid    = config["use_mid"]
    use_QRNN   = config["use_QRNN"]
    use_50NN   = config["use_50NN"]
    use_MSENN  = config["use_MSENN"]
    use_CEM    = config["use_CEM"]
    use_sampling = config["use_sampling"]
    do_QRNN_step_rnd = config["do_QRNN_step_rnd"]
    do_RS      = config["do_RS"]
    
    delta_t = 0.05 # [sec]
    sim_steps = 150 # [steps]
    print(f"[INFO] delta_t : {delta_t:.2f}[s] , sim_steps : {sim_steps}[steps], total_sim_time : {delta_t*sim_steps:.2f}[s]")

    # initialize a pendulum as a control target
    pendulum = Pendulum(
        mass_of_pole = 1.0,
        length_of_pole = 1.0,
        max_torque_abs = 2.0,
        max_speed_abs = 8.0,
        delta_t = delta_t,
        visualize = True,
    )
    pendulum.reset(
        init_state = np.array([np.pi, 0.0]), # [theta(rad), theta_dot(rad/s)]
    )

    num_quantiles = 11
    state_dim = len(pendulum.state)
    goal_state_dim = state_dim
    action_dim = 1

    if use_QRNN:
        
        model_state = NextStateQuantileNetwork(state_dim, action_dim, num_quantiles)
        # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
        # optimizer_state = optimizer_QRNN
        loss_state = quantile_loss
        # replay_buffer_state = replay_buffer_QRNN

    elif use_50NN:
        model_state = NextStateSinglePredNetwork(state_dim, action_dim)
        # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
        # optimizer_state = optimizer_50NN
        loss_state = quantile_loss_median
        # replay_buffer_state = replay_buffer_50NN

    elif use_MSENN:
        model_state = NextStateSinglePredNetwork(state_dim, action_dim)
        # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
        # optimizer_state = optimizer_MSENN
        loss_state = mse_loss
        # replay_buffer_state = replay_buffer_MSENN

    optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
    replay_buffer_state = [] # Experience replay buffer

    if use_ASNN:
        replay_buffer_ASNN = ReplayBuffer_ASNN(10000)
        model_ASNN = ActionSequenceNN(state_dim, goal_state_dim, action_dim)
        optimizer_ASNN = optim.Adam(model_ASNN.parameters(), lr=1e-3)

    else:
        replay_buffer_ASNN = None
        model_ASNN = None
        optimizer_ASNN = None
        
    prob = "Pendulum_TrueMPC"
    prob_name = "Pendulum"

    time_horizon = 20

    batch_size = 32
    nb_MPC_iters = 5
    num_particles = 100
    # stage_cost_weight = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
    # terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
    stage_cost_weight = torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]
    terminal_cost_weight = 5.0 * torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]

    prob_vars = setup_class(pendulum, prob_name, delta_t, sim_steps, stage_cost_weight, terminal_cost_weight, do_RS, use_sampling, use_mid, use_QRNN, use_50NN, use_MSENN, model_state, optimizer_state, loss_state, replay_buffer_state, use_ASNN, model_ASNN, loss_ASNN, replay_buffer_ASNN, use_CEM, num_quantiles)

    # Generate new uniformly random action sequences at each step in the env
    particles = generate_random_action_sequences(prob_vars)
    print("particles ", particles.shape, "\n")

    theta = []
    theta_dot = []

    # simulation loop
    for i in range(sim_steps):

        # get current state of pendulum
        current_state = pendulum.get_state()
        theta.append(current_state[0])
        theta_dot.append(current_state[1])

        if use_QRNN:
            input_torque, input_torque_sequence, best_cost, particles = choose_action_func_QRNN(prob_vars, current_state, particles)

        if use_50NN or use_MSENN:
            input_torque, input_torque_sequence, best_cost, particles = choose_action_func_50NN_MSENN(prob_vars, current_state, particles)

        # print current state and input torque
        print("current_state ", current_state, "\n")
        print("input_torque ", input_torque, "\n")
        print(f"Time: {i*delta_t:>2.2f}[s], theta={current_state[0]:>+3.3f}[rad], theta_dot={current_state[1]:>+3.3f}[rad/s], input torque={input_torque:>+3.2f}[Nm]", end="")
        print(", # currently staying upright #" if abs(current_state[0]) < 0.1 and abs(current_state[1] < 0.1) else "")

        # update states of pendulum
        pendulum.update(u=[input_torque], delta_t=delta_t)

        next_state = pendulum.get_state()

        # replay_buffer_50NN.append((current_state, np.array([input_torque]), next_state))
        replay_buffer_state.append((current_state, np.array([input_torque]), next_state))

        # replay_buffer_ASNN.push(current_state, goal_state, np.array([input_torque]))
        
        if len(replay_buffer_state) < batch_size:
            pass
        else:
            if use_QRNN:
                train_QRNN(prob_vars, prob_vars.model_state, prob_vars.replay_buffer_state, prob_vars.optimizer_state)
            
            if use_50NN or use_MSENN:
                train_50NN_MSENN(prob_vars, prob_vars.model_state, prob_vars.replay_buffer_state, prob_vars.optimizer_state, prob_vars.loss_state)

        if not do_RS or not do_QRNN_step_rnd: # basic method: Shift particles to the left and add new actions sampled for a uniform distribution
                    
            particles = ShiftParticlesReplaceWithRandom_func(prob_vars, particles)
                    
        particles = np.clip(particles, -pendulum.max_torque, pendulum.max_torque)
        
    # show animation
    pendulum.show_animation(interval_ms=int(delta_t * 1000))
    # save animation
    # pendulum.save_animation(f"{method_name}_{prob_name}.mp4", interval=int(delta_t * 1000))
    # pendulum.save_animation("mppi_pendulum.mp4", interval=int(delta_t * 1000), movie_writer="ffmpeg") # ffmpeg is required to write mp4 file

    auc_theta_mean, auc_theta_std = compute_auc_and_std(theta, sim_steps)
    auc_theta_dot_mean, auc_theta_dot_std = compute_auc_and_std(theta_dot, sim_steps)

    print(f"{method_name} - theta: AUC = {auc_theta_mean:.2f} ± {auc_theta_std:.2f} \n")
    print(f"{method_name} - theta_dot: AUC = {auc_theta_dot_mean:.2f} ± {auc_theta_dot_std:.2f} \n")

    pendulum_theta_dict[method_name] = theta
    pendulum_theta_dot_dict[method_name] = theta_dot

    pendulum_theta_dict_auc_mean[method_name] = auc_theta_mean
    pendulum_theta_dict_auc_std[method_name] = auc_theta_std
    pendulum_theta_dot_dict_auc_mean[method_name] = auc_theta_dot_mean
    pendulum_theta_dot_dict_auc_std[method_name] = auc_theta_dot_std



Simulation settings


In [ ]:
delta_t = 0.05 # [sec]
sim_steps = 150 # [steps]
print(f"[INFO] delta_t : {delta_t:.2f}[s] , sim_steps : {sim_steps}[steps], total_sim_time : {delta_t*sim_steps:.2f}[s]")

# initialize a pendulum as a control target
pendulum = Pendulum(
    mass_of_pole = 1.0,
    length_of_pole = 1.0,
    max_torque_abs = 2.0,
    max_speed_abs = 8.0,
    delta_t = delta_t,
    visualize = True,
)
pendulum.reset(
    init_state = np.array([np.pi, 0.0]), # [theta(rad), theta_dot(rad/s)]
)

num_quantiles = 11
state_dim = len(pendulum.state)
goal_state_dim = state_dim
action_dim = 1

if use_QRNN:
    
    model_state = NextStateQuantileNetwork(state_dim, action_dim, num_quantiles)
    # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
    # optimizer_state = optimizer_QRNN
    loss_state = quantile_loss
    # replay_buffer_state = replay_buffer_QRNN

elif use_50NN:
    model_state = NextStateSinglePredNetwork(state_dim, action_dim)
    # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
    # optimizer_state = optimizer_50NN
    loss_state = quantile_loss_median
    # replay_buffer_state = replay_buffer_50NN

elif use_MSENN:
    model_state = NextStateSinglePredNetwork(state_dim, action_dim)
    # optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
    # optimizer_state = optimizer_MSENN
    loss_state = mse_loss
    # replay_buffer_state = replay_buffer_MSENN

optimizer_state = optim.Adam(model_state.parameters(), lr=1e-3)
replay_buffer_state = [] # Experience replay buffer

if use_ASNN:
    replay_buffer_ASNN = ReplayBuffer_ASNN(10000)
    model_ASNN = ActionSequenceNN(state_dim, goal_state_dim, action_dim)
    optimizer_ASNN = optim.Adam(model_ASNN.parameters(), lr=1e-3)

else:
    replay_buffer_ASNN = None
    model_ASNN = None
    optimizer_ASNN = None


What to run


In [ ]:
prob = "Pendulum_TrueMPC"

# Run Random shooting (RS)
# do_RS = True
# use_ASNN = False
# use_sampling = False
# use_mid = True
# do_QRNN_step_rnd = False
# method_name = "RS_mid_50NN"
# use_QRNN = False
# use_50NN = True
# use_MSENN = False

# use_CEM = False # Use PF
# use_PF = True # Use CEM



# model_50NN = NextStateSinglePredNetwork(state_dim, action_dim)
# optimizer_50NN = optim.Adam(model_50NN.parameters(), lr=1e-3)
# loss_50NN = quantile_loss_median

# # Experience replay buffer
# replay_buffer_50NN = []



time_horizon = 20

batch_size = 32
nb_MPC_iters = 5
num_particles = 100
# stage_cost_weight = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
# terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
stage_cost_weight = torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]
terminal_cost_weight = 5.0 * torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]

prob_name = "Pendulum"

# loss_ASNN = None

prob_vars = setup_class(pendulum, prob_name, delta_t, sim_steps, stage_cost_weight, terminal_cost_weight, do_RS, use_sampling, use_mid, use_QRNN, use_50NN, use_MSENN, model_state, optimizer_state, loss_state, replay_buffer_state, use_ASNN, model_ASNN, loss_ASNN, replay_buffer_ASNN, use_CEM, num_quantiles)

# mppi = MPPIControllerForPendulum(
#     delta_t = delta_t,
#     mass_of_pole = 1.0,
#     length_of_pole = 1.0,
#     max_torque_abs = 2.0,
#     max_speed_abs = 8.0,
#     horizon_step_T = 20,
#     number_of_samples_K = 2000,
#     param_exploration = 0.05,
#     param_lambda = 0.5,
#     param_alpha = 0.8,
#     sigma = 1.0,
#     stage_cost_weight    = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
#     terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
# )

# Generate new uniformly random action sequences at each step in the env
particles = generate_random_action_sequences(prob_vars)
print("particles ", particles.shape, "\n")

theta = []
theta_dot = []

# simulation loop
for i in range(sim_steps):

    # get current state of pendulum
    current_state = pendulum.get_state()
    theta.append(current_state[0])
    theta_dot.append(current_state[1])

    # calculate input force with MPPI
    # input_torque, input_torque_sequence = mppi.calc_control_input(
    #     observed_x = current_state
    # )

    if use_QRNN:
        input_torque, input_torque_sequence, best_cost, particles = choose_action_func_QRNN(prob_vars, current_state, particles)

    if use_50NN or use_MSENN:
        input_torque, input_torque_sequence, best_cost, particles = choose_action_func_50NN_MSENN(prob_vars, current_state, particles)

    # print current state and input torque
    print("current_state ", current_state, "\n")
    print("input_torque ", input_torque, "\n")
    print(f"Time: {i*delta_t:>2.2f}[s], theta={current_state[0]:>+3.3f}[rad], theta_dot={current_state[1]:>+3.3f}[rad/s], input torque={input_torque:>+3.2f}[Nm]", end="")
    print(", # currently staying upright #" if abs(current_state[0]) < 0.1 and abs(current_state[1] < 0.1) else "")

    # update states of pendulum
    pendulum.update(u=[input_torque], delta_t=delta_t)

    next_state = pendulum.get_state()

    # replay_buffer_50NN.append((current_state, np.array([input_torque]), next_state))
    replay_buffer_state.append((current_state, np.array([input_torque]), next_state))

    # replay_buffer_ASNN.push(current_state, goal_state, np.array([input_torque]))
    
    if len(replay_buffer_state) < batch_size:
        pass
    else:
        if use_QRNN:
            train_QRNN(prob_vars, prob_vars.model_state, prob_vars.replay_buffer_state, prob_vars.optimizer_state)
        
        if use_50NN or use_MSENN:
            train_50NN_MSENN(prob_vars, prob_vars.model_state, prob_vars.replay_buffer_state, prob_vars.optimizer_state, prob_vars.loss_state)

    if not do_RS or not do_QRNN_step_rnd: # basic method: Shift particles to the left and add new actions sampled for a uniform distribution
                
        particles = ShiftParticlesReplaceWithRandom_func(prob_vars, particles)
                
    particles = np.clip(particles, -pendulum.max_torque, pendulum.max_torque)
    

# show animation
pendulum.show_animation(interval_ms=int(delta_t * 1000))
# save animation
# pendulum.save_animation(f"{method_name}_{prob_name}.mp4", interval=int(delta_t * 1000))
# pendulum.save_animation("mppi_pendulum.mp4", interval=int(delta_t * 1000), movie_writer="ffmpeg") # ffmpeg is required to write mp4 file


In [ ]:
def compute_auc_and_std(data, nb_episodes):
    # data = np.load(filepath, allow_pickle=True)
    # all_returns = data['episode_rewards']  # shape: (n_seeds, n_episodes)
    # print("all_returns ", all_returns, "\n")
    
    # print("all_returns shape:", all_returns.shape, "\n")
    # aucs = []
    # for rewards in all_returns:
    #     returns = rewards[:nb_episodes]
    #     x = np.arange(len(returns))
    #     aucs.append(auc(x, returns))
    
    x = np.arange(len(data))
    aucs = auc(x,data)

    auc_mean = np.mean(aucs)
    auc_std = np.std(aucs)
    return auc_mean, auc_std

pendulum_theta_dict_auc_mean = {}
pendulum_theta_dict_auc_std = {}
pendulum_theta_dot_dict_auc_mean = {}
pendulum_theta_dot_dict_auc_std = {}

auc_theta_mean, auc_theta_std = compute_auc_and_std(theta, sim_steps)
auc_theta_dot_mean, auc_theta_dot_std = compute_auc_and_std(theta_dot, sim_steps)

print(f"{method_name} - theta: AUC = {auc_theta_mean:.2f} ± {auc_theta_std:.2f} \n")
print(f"{method_name} - theta_dot: AUC = {auc_theta_dot_mean:.2f} ± {auc_theta_dot_std:.2f} \n")

# print(f"{label}: AUC = {auc_mean:.2f} ± {auc_std:.2f}")
# pendulum_theta_dict_auc_mean[label] = auc_mean
# pendulum_theta_dict_auc_std[label] = auc_std

    

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
plt.plot(np.arange(0, sim_steps*delta_t, delta_t), np.rad2deg(theta), label="theta [deg]")
plt.xlabel("time [s]")
plt.ylabel("angle [deg]")
plt.legend()

In [ ]:
plt.plot(np.arange(0, sim_steps*delta_t, delta_t), np.rad2deg(theta_dot), label="theta_dot [deg/s]")
plt.xlabel("time [s]")
plt.ylabel("angular velocity [deg/s]")
# plt.legend()


In [ ]:
plt.plot()

Working code


In [ ]:
# simulation settings
delta_t = 0.05 # [sec]
sim_steps = 150 # [steps]
print(f"[INFO] delta_t : {delta_t:.2f}[s] , sim_steps : {sim_steps}[steps], total_sim_time : {delta_t*sim_steps:.2f}[s]")

# initialize a pendulum as a control target
pendulum = Pendulum(
    mass_of_pole = 1.0,
    length_of_pole = 1.0,
    max_torque_abs = 2.0,
    max_speed_abs = 8.0,
    delta_t = delta_t,
    visualize = True,
)
pendulum.reset(
    init_state = np.array([np.pi, 0.0]), # [theta(rad), theta_dot(rad/s)]
)

prob = "Pendulum_TrueMPC"

# Run Random shooting (RS)
do_RS = True
use_ASNN = False
use_sampling = False
use_mid = True
do_QRNN_step_rnd = False
method_name = "RS_mid_50NN"
use_QRNN = False
use_50NN = True
use_MSENN = False

use_CEM = False # Use PF
# use_PF = True # Use CEM

state_dim = len(pendulum.state)
action_dim = 1

model_50NN = NextStateSinglePredNetwork(state_dim, action_dim)
optimizer_50NN = optim.Adam(model_50NN.parameters(), lr=1e-3)
loss_50NN = quantile_loss_median

# Experience replay buffer
replay_buffer_50NN = []

replay_buffer_ASNN = None
model_ASNN = None
optimizer_ASNN = None

time_horizon = 20

batch_size = 32
nb_MPC_iters = 5
num_particles = 100
# stage_cost_weight = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
# terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
stage_cost_weight = torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]
terminal_cost_weight = 5.0 * torch.tensor([1.0, 0.1]) # weight for [theta, theta_dot]

prob_name = "Pendulum"

loss_ASNN = None

prob_vars = setup_class(pendulum, prob_name, delta_t, sim_steps, stage_cost_weight, terminal_cost_weight, do_RS, use_sampling, use_mid, use_QRNN, use_50NN, use_MSENN, model_50NN, optimizer_50NN, loss_50NN, replay_buffer_50NN, use_ASNN, model_ASNN, loss_ASNN, replay_buffer_ASNN, use_CEM)

# mppi = MPPIControllerForPendulum(
#     delta_t = delta_t,
#     mass_of_pole = 1.0,
#     length_of_pole = 1.0,
#     max_torque_abs = 2.0,
#     max_speed_abs = 8.0,
#     horizon_step_T = 20,
#     number_of_samples_K = 2000,
#     param_exploration = 0.05,
#     param_lambda = 0.5,
#     param_alpha = 0.8,
#     sigma = 1.0,
#     stage_cost_weight    = np.array([1.0, 0.1]), # weight for [theta, theta_dot]
#     terminal_cost_weight = 5.0 * np.array([1.0, 0.1]), # weight for [theta, theta_dot]
# )

# Generate new uniformly random action sequences at each step in the env
particles = generate_random_action_sequences(prob_vars)
print("particles ", particles.shape, "\n")

theta = []
theta_dot = []

# simulation loop
for i in range(sim_steps):

    # get current state of pendulum
    current_state = pendulum.get_state()
    theta.append(current_state[0])
    theta_dot.append(current_state[1])

    # calculate input force with MPPI
    # input_torque, input_torque_sequence = mppi.calc_control_input(
    #     observed_x = current_state
    # )

    input_torque, input_torque_sequence, best_cost, particles = choose_action_func_50NN_MSENN(prob_vars, current_state, particles)

    # print current state and input torque
    print(f"Time: {i*delta_t:>2.2f}[s], theta={current_state[0]:>+3.3f}[rad], theta_dot={current_state[1]:>+3.3f}[rad/s], input torque={input_torque:>+3.2f}[Nm]", end="")
    print(", # currently staying upright #" if abs(current_state[0]) < 0.1 and abs(current_state[1] < 0.1) else "")

    # update states of pendulum
    pendulum.update(u=[input_torque], delta_t=delta_t)

    next_state = pendulum.get_state()

    replay_buffer_50NN.append((current_state, np.array([input_torque]), next_state))
    # replay_buffer_ASNN.push(current_state, goal_state, np.array([input_torque]))
    
    if len(replay_buffer_50NN) < batch_size:
        pass
    else:
        train_50NN_MSENN(prob_vars, prob_vars.model_state, prob_vars.replay_buffer_state, prob_vars.optimizer_state, prob_vars.loss_state)

    if not do_RS or not do_QRNN_step_rnd: # basic method: Shift particles to the left and add new actions sampled for a uniform distribution
                
        particles = ShiftParticlesReplaceWithRandom_func(prob_vars, particles)
                
    particles = np.clip(particles, -pendulum.max_torque, pendulum.max_torque)
    

# show animation
pendulum.show_animation(interval_ms=int(delta_t * 1000))
# save animation
# pendulum.save_animation(f"{method_name}_{prob_name}.mp4", interval=int(delta_t * 1000))
# pendulum.save_animation("mppi_pendulum.mp4", interval=int(delta_t * 1000), movie_writer="ffmpeg") # ffmpeg is required to write mp4 file

In [ ]:
pendulum_theta_dict_auc_mean = {}
pendulum_theta_dict_auc_std = {}
pendulum_theta_dot_dict_auc_mean = {}
pendulum_theta_dot_dict_auc_std = {}

auc_theta_mean, auc_theta_std = compute_auc_and_std(theta, sim_steps)
auc_theta_dot_mean, auc_theta_dot_std = compute_auc_and_std(theta_dot, sim_steps)

print(f"{method_name} - theta: AUC = {auc_theta_mean:.2f} ± {auc_theta_std:.2f} \n")
print(f"{method_name} - theta_dot: AUC = {auc_theta_dot_mean:.2f} ± {auc_theta_dot_std:.2f} \n")
